## PyTorch Autograd: Automatic Differentiation

- **Autograd** is PyTorch’s built-in automatic differentiation engine. It powers neural network training by automatically calculating the gradients (derivatives) of a computation with respect to its inputs.
- When training a neural network, algorithms like gradient descent need to know how much each weight and bias contributed to the final error. Autograd calculates this backward step automatically.

#### Core Concepts

* **`requires_grad=True`**: When you create a tensor and set this attribute to `True`, PyTorch starts tracking every operation performed on that tensor.
* **Computational Graph**: PyTorch builds a Directed Acyclic Graph (DAG) under the hood. Tensors are the nodes, and the mathematical operations (functions) are the edges.
* **`.backward()`**: When you finish your computation, you call this method on the final tensor (usually the loss). Autograd then traverses the graph backward to compute all gradients.
* **`.grad`**: The computed gradients are stored in this attribute of the original tensors.

## The Mathematics of Autograd: The Chain Rule

At its core, PyTorch Autograd is an engine that automatically applies the **Chain Rule** from calculus. Let's look at a composite function:

1. $y = x^2$
2. $z = \sin(y)$
3. $u = e^z$

If we want to find how a change in our initial input $x$ affects our final output $u$, we need to calculate the derivative of $u$ with respect to $x$, denoted as $\frac{du}{dx}$.

Because $u$ is built from a sequence of functions, we use the Chain Rule, which states:


$$\frac{du}{dx} = \frac{du}{dz} \cdot \frac{dz}{dy} \cdot \frac{dy}{dx}$$

Let's calculate the local derivatives for each step:

* **Step 1:** $\frac{dy}{dx} = 2x$
* **Step 2:** $\frac{dz}{dy} = \cos(y)$
* **Step 3:** $\frac{du}{dz} = e^z$

Combining them gives us our final analytical gradient:


$$\frac{du}{dx} = e^z \cdot \cos(y) \cdot 2x$$

---

In [1]:
import math
import torch

#### 1. MANUAL CALCULATION (Without Autograd)


In [2]:
x_val = 2.0
y_val = x_val ** 2
z_val = math.sin(y_val)
u_val = math.exp(z_val)

# Backward Pass (Chain Rule)
dy_dx = 2 * x_val
dz_dy = math.cos(y_val)
du_dz = math.exp(z_val)
du_dx = du_dz * dz_dy * dy_dx

print(f"Manual Gradient (du/dx): {du_dx:.4f}\n")

Manual Gradient (du/dx): -1.2267



#### 2. PYTORCH AUTOGRAD CALCULATION

In [3]:
x = torch.tensor(2.0, requires_grad=True)

# Forward Pass (Graph is built)
y = x ** 2
z = torch.sin(y)
u = torch.exp(z)

# Backward Pass (Chain rule is automated)
u.backward()

print(f"Autograd Gradient (du/dx): {x.grad.item():.4f}")

Autograd Gradient (du/dx): -1.2267


### The Mathematical Model (Logistic Regression)

1. **Linear Transformation:** Computes the weighted sum of inputs plus a bias.

$$z = w \cdot x + b$$


2. **Activation (Sigmoid Function):** Squeezes the linear output into a probability between 0 and 1.

$$y_{\text{pred}} = \sigma(z) = \frac{1}{1 + e^{-z}}$$


3. **Loss Function (Binary Cross-Entropy Loss):** Compares the predicted probability to the actual target.

$$L = - [y_{\text{target}} \cdot \ln(y_{\text{pred}}) + (1 - y_{\text{target}}) \cdot \ln(1 - y_{\text{pred}})]$$

<img alt="Logo" caption="Pytorch logo" src="./Image/simple_network.png"> 

In [4]:
# import torch

# Inputs
x = torch.tensor(6.7)  # Input feature
y = torch.tensor(0.0)  # True label (binary)

w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias

In [5]:
# Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8  # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

In [6]:
# Forward pass
z = w * x + b  # Weighted sum (linear part)
y_pred = torch.sigmoid(z)  # Predicted probability

# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

In [7]:
loss

tensor(6.7012)

In [8]:
# Derivatives:
# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x  # dz/dw = x
dz_db = 1  # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

In [9]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


In [10]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)

In [11]:
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

In [12]:
w

tensor(1., requires_grad=True)

In [13]:
b

tensor(0., requires_grad=True)

In [14]:
z = w*x + b
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [15]:
y_pred = torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [16]:
loss = binary_cross_entropy_loss(y_pred, y)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [17]:
loss.backward()

In [18]:
print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


In [19]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

In [20]:
x

tensor([1., 2., 3.], requires_grad=True)

In [21]:
y = (x**2).mean()
y

tensor(4.6667, grad_fn=<MeanBackward0>)

In [22]:
y.backward()

In [23]:
x.grad

tensor([0.6667, 1.3333, 2.0000])

### Stopping Autograd (Inference Mode)

You don't always want PyTorch to track gradients. Tracking operations consumes memory and computational power. When you are evaluating a model, making predictions (inference), or fine-tuning only specific layers, you should turn Autograd off.

**Ways to stop tracking:**

1. **`torch.no_grad()`**: A context manager that disables gradient calculation for a block of code.
2. **`.detach()`**: Creates a new tensor that shares the same data but is detached from the computational graph.

> **Important Note:** PyTorch accumulates gradients. If you run `.backward()` multiple times in a loop (like during training), the new gradients are added to the old ones. You must clear them using `optimizer.zero_grad()` or `tensor.grad.zero_()` before the next pass.

---

In [24]:
# clearing grad
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [25]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

In [26]:
y.backward()

In [27]:
x.grad

tensor(4.)

In [28]:
x.grad.zero_()

tensor(0.)

In [29]:
# disable gradient tracking
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [30]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

In [31]:
y.backward()

In [32]:
x.grad

tensor(4.)

In [33]:
# option 1 - requires_grad_(False)
# option 2 - detach()
# option 3 - torch.no_grad()

In [34]:
x.requires_grad_(False)

tensor(2.)

In [35]:
x

tensor(2.)

In [36]:
y = x ** 2

In [37]:
y

tensor(4.)

In [38]:
y.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
x

In [39]:
z = x.detach()
z

tensor(2.)

In [40]:
y = x ** 2

In [41]:
y

tensor(4.)

In [42]:
y1 = z ** 2
y1

tensor(4.)

In [43]:
y.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [44]:
y1.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [45]:
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [46]:
y = x ** 2

In [47]:
y

tensor(4., grad_fn=<PowBackward0>)

In [48]:
y.backward()

In [49]:
x = torch.tensor([1.], requires_grad=True)
with torch.no_grad():
    y = x * 2

print(y.requires_grad)
# Output: False


False
